<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo:Failure Diagnosis Workflow_sol</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Demo: Failure Diagnosis Workflow

**Purpose:**  
This notebook teaches how engineers systematically analyze and debug failures in AI agent systems by reading execution traces, identifying the root cause, adjusting prompts or tool descriptions, and validating the improved behavior.

In real production environments, agent failures can cause serious consequences such as production downtime, quality defects, safety risks, and financial losses. Learning how to diagnose these failures quickly is a critical skill.

## 1. Setup

In [1]:
# Setup - Run this first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*" beautifulsoup4 requests --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 497.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/

## 2. Tools for Predictive Maintenance Agent

These tools simulate real factory data sources used in manufacturing plants for predictive maintenance.

In [2]:
from langchain_core.tools import tool

@tool
def check_machine_status(machine_id: str) -> str:
    """Check current operational status of a manufacturing machine."""
    status_db = {
        "CNC-01": "Vibration level high. Temperature normal. Running at 85% capacity.",
        "ROBOT-03": "Normal operation. No anomalies detected.",
        "CONVEYOR-02": "Motor temperature elevated. Potential bearing issue."
    }
    return status_db.get(machine_id, f"No data available for machine {machine_id}.")

@tool
def get_maintenance_history(machine_id: str) -> str:
    """Retrieve past maintenance records for a machine."""
    history = {
        "CNC-01": "Last maintenance: 12 days ago. Bearing replaced twice this year.",
        "ROBOT-03": "Last maintenance: 45 days ago. No major issues.",
        "CONVEYOR-02": "Last maintenance: 8 days ago. Motor overheating reported."
    }
    return history.get(machine_id, "No maintenance history found.")

@tool
def predict_failure_risk(machine_id: str) -> str:
    """Predict failure risk based on current and historical data."""
    risk_db = {
        "CNC-01": "High risk of failure within 48 hours due to vibration patterns.",
        "ROBOT-03": "Low risk. System stable.",
        "CONVEYOR-02": "Medium risk. Monitor motor temperature closely."
    }
    return risk_db.get(machine_id, "Risk assessment not available.")

## 3. Broken Agent Setup

We intentionally create a broken version of the Predictive Maintenance Agent to demonstrate common failure patterns that occur in real industrial environments.

In [3]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

broken_prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

CRITICAL RULE: For any machine status or risk question, always start with predict_failure_risk tool first.

Use the following format exactly:

Thought: ...
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (repeat if needed)
Thought: I now know the final answer
Final Answer: the final answer

Begin!

Question: {input}
{agent_scratchpad}"""

broken_prompt = PromptTemplate.from_template(broken_prompt_template)

broken_agent = create_react_agent(llm=llm, tools=[check_machine_status, get_maintenance_history, predict_failure_risk], prompt=broken_prompt)

broken_executor = AgentExecutor(
    agent=broken_agent,
    tools=[check_machine_status, get_maintenance_history, predict_failure_risk],
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

print("Broken Predictive Maintenance Agent is ready for diagnosis")

Broken Predictive Maintenance Agent is ready for diagnosis


## 4. Running the Broken Agent

Let's run a realistic factory query and observe the failures in the execution trace.

In [4]:
query = "Check the status of CNC-01 machine and predict its failure risk."

print("Running broken agent with query:")
response = broken_executor.invoke({"input": query})
print("\nFinal Output:", response.get("output", "No output"))

Running broken agent with query:


> Entering new AgentExecutor chain...
Thought: I should start by using the predict_failure_risk tool to get the failure risk prediction for CNC-01 machine.
Action: predict_failure_risk
Action Input: CNC-01High risk of failure within 48 hours due to vibration patterns.I should now check the current operational status of CNC-01 machine to see if it aligns with the predicted failure risk.
Action: check_machine_status
Action Input: CNC-01Vibration level high. Temperature normal. Running at 85% capacity.Based on the high risk of failure prediction and the high vibration level, it seems like CNC-01 machine is indeed at risk of failure.
Final Answer: CNC-01 machine has a high risk of failure.

> Finished chain.

Final Output: CNC-01 machine has a high risk of failure.


## 5. Reading Execution Traces

In real manufacturing plants, engineers spend significant time reading agent execution traces to diagnose problems.

**Key Elements to Look For in Every Trace:**

- **Thought**: What the agent is thinking and planning at each step.
- **Action**: Which tool the agent decided to call.
- **Action Input**: What parameters it passed to the tool.
- **Observation**: What the tool returned (success or error).
- **Final Answer**: The conclusion the agent gave to the user.

By carefully analyzing these steps, we can pinpoint exactly where the agent went wrong.

## 6. Identifying Failure Points in the Agent Loop

When we examine the trace from the broken agent, we can identify several common failure types that occur in real industrial agent systems.

**General Issues Caused by These Failures:**

- **Wrong Tool Selection**  
  The agent chooses an inappropriate tool. In manufacturing, this can lead to checking the wrong sensor or using the wrong diagnostic method, resulting in missed failures, delayed maintenance, and unexpected production downtime.

- **Repeated Loops**  
  The agent keeps calling tools in a loop without making progress. This wastes computational resources, increases latency, and can delay critical decisions in time-sensitive factory operations.

- **Incorrect Reasoning Steps / Premature Answer**  
  The agent jumps to conclusions with insufficient data or without proper verification. In a factory environment, this can lead to wrong maintenance decisions, unnecessary shutdowns, or — worse — missing a real safety or quality issue.

These failures are not just technical problems — they have direct business and safety consequences.

## 7. Fixing the Agent – Adjusting Prompt and Tool Descriptions

Now we will fix the issues by improving the prompt and tool descriptions.

**General Fix Strategies:**
- Make instructions more specific and ordered
- Add explicit reasoning steps
- Improve tool descriptions to reduce ambiguity
- Add safety constraints and cautious language

In [5]:
fixed_prompt_template = """You are a Predictive Maintenance Agent for a smart factory.

Follow this reasoning order strictly:
1. First check the current machine status using check_machine_status tool.
2. Then check maintenance history if needed.
3. Finally predict failure risk using predict_failure_risk tool.

Always be cautious. When risk is detected, recommend further monitoring or inspection.

{tools}

Use the following format exactly:

Thought: ...
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (repeat if needed)
Thought: I now know the final answer
Final Answer: the final answer

Begin!

Question: {input}
{agent_scratchpad}"""

fixed_prompt = PromptTemplate.from_template(fixed_prompt_template)

fixed_agent = create_react_agent(llm=llm, tools=[check_machine_status, get_maintenance_history, predict_failure_risk], prompt=fixed_prompt)

fixed_executor = AgentExecutor(
    agent=fixed_agent,
    tools=[check_machine_status, get_maintenance_history, predict_failure_risk],
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True
)

print(" Fixed Agent created. Now let's test the improvement.")

 Fixed Agent created. Now let's test the improvement.


## 8. Validating Improved Behavior

Let's run the same query with the fixed agent and compare the trace and final output.

In [6]:
print("Running fixed agent with the same query:")
fixed_response = fixed_executor.invoke({"input": query})
print("\nFinal Output (Fixed):", fixed_response.get("output", "No output"))

Running fixed agent with the same query:


> Entering new AgentExecutor chain...
Thought: I should first check the current status of CNC-01 machine.
Action: check_machine_status
Action Input: machine_id = "CNC-01"No data available for machine machine_id = "CNC-01.I need to retrieve the maintenance history for CNC-01 machine to get more insights.
Action: get_maintenance_history
Action Input: machine_id = "CNC-01"No maintenance history found.Since there is no current status or maintenance history available for CNC-01 machine, I should predict its failure risk based on other data.
Action: predict_failure_risk
Action Input: machine_id = "CNC-01"Risk assessment not available.I now know the final answer.
Final Answer: Insufficient data available to predict failure risk for CNC-01 machine. Further monitoring or inspection is recommended.

> Finished chain.

Final Output (Fixed): Insufficient data available to predict failure risk for CNC-01 machine. Further monitoring or inspection is recomme

## Conclusion

In this demo, we learned a systematic approach to diagnose and fix agent failures:

1. Run the agent and capture the full execution trace
2. Identify the failure point by reading Thought → Action → Observation
3. Adjust the prompt or tool description to address the root cause
4. Re-test and validate the improvement

**General Impact of These Failures in Industry:**
- Wrong tool selection can lead to missed maintenance and costly downtime.
- Repeated loops waste resources and delay critical decisions.
- Incorrect reasoning can result in wrong actions or safety risks.

Mastering failure diagnosis through trace analysis is one of the most valuable skills when building reliable AI agents in manufacturing or any other industry.

**Practice Task:**  
Go back to each failure case, fix the prompt or tool, and compare the traces before and after your changes.

## Conclusion

This demo showed three common agent failure cases in a manufacturing context:

1. Wrong tool selection
2. Repeated loops
3. Incorrect reasoning steps

**Key Takeaway:**  
Carefully inspecting execution traces is the most effective way to identify and fix agent failures before deployment.